# Tile classifier evaluation

Decides whether a retrained checkpoint is better than the baseline. The headline is **false
alarms per driving frame**, because the system raises alerts on frames. Everything else
checks that the quieter model did not lose real cracks. Logic lives in `eval/classifier.py`,
`eval/provenance.py`, `tools/eval_classifier.py` and `tools/eval_frames.py`, which are tested.

**Data rule:** run outputs can contain RDD2022 and nuScenes images. Clear all outputs before
committing (Kernel, Restart and Clear Outputs). A test fails if a committed notebook has
outputs, machine-specific paths or function definitions.

## Settings

The first checkpoint is the baseline. Thresholds come from `configs/default.yaml`.
Run with the real models: set `USE_STUBS=0` before starting Jupyter.

In [ ]:
import os
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DERIVED = REPO / "data/rdd2022/derived"
NUSCENES = REPO / "data/nuscenes"
RUNS = Path(os.environ.get("EVAL_RUNS_DIR", REPO.parent / "runs"))
CHECKPOINTS = {
    "baseline": RUNS / "tile-fold0/fold0/best.pt",
    "hard-negatives": RUNS / "tile-fold0-hn1/fold0/best.pt",
}
TRAIN_CONFIG = DERIVED / "defects_tile_train.local.yaml"
PROBE = DERIVED / "probe_negatives_batch3.json"  # confirmed clean tiles kept out of training
REVIEW_KEY = NUSCENES / "derived/annotation_key_nuscenes1/key.json"
REVIEW_CALLS = NUSCENES / "derived/annotation_nuscenes1/annotations_nuscenes1.csv"
EXPORT_DIR = NUSCENES / "derived/annotation_nuscenes2"  # never-reviewed alerts, for annotation
EXPORT_KEY_DIR = NUSCENES / "derived/annotation_key_nuscenes2"
REPORT = NUSCENES / "derived/classifier_report.json"

## Run

Scores validation tiles, the clean probe tiles and every nuScenes CAM_FRONT frame for each
checkpoint. The road mask and object detector run once per frame and are shared, so only the
classifier differs. Slow: several minutes on a GPU.

In [ ]:
import json

from tools.eval_classifier import main

main(
    [
        "--train-config",
        str(TRAIN_CONFIG),
        *[f"--checkpoint={name}={path}" for name, path in CHECKPOINTS.items()],
        "--probe-negatives",
        str(PROBE),
        "--nuscenes-root",
        str(NUSCENES / "raw"),
        "--review-key",
        str(REVIEW_KEY),
        "--review-calls",
        str(REVIEW_CALLS),
        "--export-dir",
        str(EXPORT_DIR),
        "--export-key-dir",
        str(EXPORT_KEY_DIR),
        "--out",
        str(REPORT),
    ]
)
report = json.loads(REPORT.read_text())
baseline = report["baseline"]
results = report["checkpoints"]

## Provenance

What these numbers were measured on. If any hash or the thresholds differ from an
older report, the two reports are not comparable.

In [ ]:
prov = report["provenance"]
print("git commit:", prov["git_commit"], "(uncommitted changes)" if prov["git_dirty"] else "")
print("date (UTC):", prov["date_utc"])
print("thresholds:", prov["thresholds"])
for name, digest in prov["files"].items():
    print(f"  {name:<28} {digest[:16]}")

## Headline: alerts per driving frame

Alerts on the 404 nuScenes mini frames through the shipped pipeline. Fewer is quieter, but see
the recall guard below: a model that finds nothing would also score well here.

In [ ]:
print(f"{'checkpoint':<18}{'alerts':>8}{'per frame':>11}{'day':>8}{'night':>8}")
for name, r in results.items():
    f = r["frames"]
    print(
        f"{name:<18}{f['alerts']:>8}{f['per_frame']:>11.3f}"
        f"{f['day']['per_frame']:>8.3f}{f['night']['per_frame']:>8.3f}"
    )

### Alerts by what the human review says

Each alerted tile is matched to your review of the baseline's alerts. **Unreviewed** tiles are
new alerts nobody has judged; they were exported to `EXPORT_DIR` as an annotation batch.
Ranges are 90% Wilson intervals on the share of each bucket.

In [ ]:
from eval.classifier import wilson_interval

print(f"{'checkpoint':<18}{'bucket':<12}{'tiles':>7}{'share of alerted tiles (90% range)':>40}")
for name, r in results.items():
    buckets = r["frames"]["buckets"]
    total = sum(buckets.values())
    for bucket, count in buckets.items():
        low, high = wilson_interval(count, total)
        print(f"{name:<18}{bucket:<12}{count:>7}{count / total:>14.0%} ({low:.0%}-{high:.0%})")

## Did we lose real cracks? (PASS/FAIL)

A checkpoint passes if no class loses more than 3 points of validation recall against the
baseline and at least 5 of the 6 reviewed cracks stay flagged. Both limits are provisional;
6 reviewed cracks is a very small sample, so a FAIL here deserves a look, not just a verdict.

In [ ]:
for name, r in results.items():
    if name == baseline:
        continue
    guard = r["guard"]
    print(f"{name}: {'PASS' if guard['passed'] else 'FAIL'}")
    for reason in guard["reasons"]:
        print("  -", reason)

## Validation tiles

Precision and recall at the shipped per-class thresholds on the fold's validation tiles.
This is the capped, roughly balanced pool, so precision here is higher than on a driving frame.
The last column is the new model's recall at the false-alarm rate the baseline has at its
threshold: higher means better, not just quieter.

In [ ]:
header = ("class", "checkpoint", "precision", "recall", "auroc", "flagged", "recall@base-FPR")
print(f"{header[0]:<6}{header[1]:<18}" + "".join(f"{h:>16}" for h in header[2:]))
for code in prov["thresholds"]:
    for name, r in results.items():
        v = r["validation"][code]
        decimals = (v["precision"], v["recall"], v["auroc"])
        print(
            f"{code:<6}{name:<18}"
            + "".join(f"{x:>16.3f}" for x in decimals)
            + f"{v['flagged']:>16}{v['recall_at_baseline_fpr']:>16.3f}"
        )

## Clean probe tiles

Annotator-confirmed clean pavement the baseline flagged, held out of training. Fewer flagged
means fewer false alarms of this kind. The set is small, so read the ranges, not the counts.

In [ ]:
for name, r in results.items():
    p = r["probe"]
    print(
        f"{name:<18} flagged {p['count']} of {p['total']}"
        f" ({p['low90']:.0%}-{p['high90']:.0%} at 90%)"
    )

## Where the alerts come from

By defect type, and by how much road the alerted tile covers.

In [ ]:
for name, r in results.items():
    f = r["frames"]
    print(name)
    print("  by defect type:", f["by_class"])
    print("  by road cover :", f["by_road_cover"])

## What the new model still gets wrong

The clean probe tiles each checkpoint still flags, most confident first. If one looks like a
real crack, the annotation was wrong, not the model.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from tools.eval_classifier import load_context, load_model, probe_crops, score_crops

ctx = load_context(TRAIN_CONFIG)
crops = probe_crops(ctx, PROBE)
limits = np.asarray(list(prov["thresholds"].values()))
for name, path in CHECKPOINTS.items():
    model, device = load_model(path, ctx)
    margin = (score_crops(model, device, crops, ctx) / limits).max(axis=1)
    shown = [i for i in np.argsort(-margin) if margin[i] >= 1.0][:12]
    fig, axes = plt.subplots(2, 6, figsize=(18, 6), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, i in zip(axes.ravel(), shown, strict=False):
        ax.imshow(crops[i])
        ax.set_title(f"{margin[i]:.2f} x threshold", fontsize=9)
    fig.suptitle(f"{name}: clean probe tiles still flagged ({len(shown)} shown)")
    plt.show()